# 🧠 Agent 실습 1 – Agent Full-Loop 수동 구현


### 학습 목표
1. **Agent의 필요성**을 일반 LLM과의 비교를 통해 체감합니다.
2. Function Calling을 이용해 **Agent의 전체 동작 사이클(Full-Loop)** 을 수동으로 구현합니다.
3. 여러 Tool이 있을 때 LLM이 어떻게 판단하는지 실험합니다.
4. **Agent가 잘못된 판단(Hallucination)을 내리는 경우**를 직접 확인합니다.
5. 수동 구현의 한계를 통해 **LangGraph 같은 프레임워크의 필요성**을 체감합니다.

---
## ⚙️ 1. 환경 설정

In [1]:
!pip install openai wikipedia requests pandas python-dotenv -q

In [2]:
import openai
import os
import json
import requests
import wikipedia
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()
MODEL = "gpt-4o-mini"

print("✅ 환경 설정 완료")

✅ 환경 설정 완료


---
## 🔧 2. Tool 정의

실습에 사용할 모든 함수(Tool)와 그 명세(JSON Schema)를 정의합니다.

이번 실습에서 사용할 Tool은 3가지입니다:
- `get_exchange_rate` – 실시간 환율 조회
- `search_wikipedia` – 위키백과 검색
- `get_current_weather` – 도시 날씨 확인 (Mock)

In [3]:
# ── Tool 함수 정의 ──────────────────────────────────────────

def get_exchange_rate(currency: str) -> str:
    """지정된 통화와 대한민국 원(KRW) 사이의 현재 환율을 가져옵니다."""
    if not isinstance(currency, str) or len(currency) != 3:
        return json.dumps({"error": "유효하지 않은 통화 코드입니다. 3자리 ISO 코드를 사용해야 합니다."})
    url = f"https://api.frankfurter.app/latest?from={currency.upper()}&to=KRW"
    try:
        data = requests.get(url, timeout=5).json()
        rate = data.get("rates", {}).get("KRW")
        if rate:
            return json.dumps({"currency": currency.upper(), "rate_to_KRW": rate})
        return json.dumps({"error": f"'{currency}'에 대한 KRW 환율 정보를 찾을 수 없습니다."})
    except Exception as e:
        return json.dumps({"error": f"API 요청 중 에러 발생: {e}"})


def search_wikipedia(query: str) -> str:
    """위키백과에서 정보를 검색하여 요약합니다."""
    try:
        wikipedia.set_lang("ko")
        return wikipedia.summary(query, sentences=2)
    except wikipedia.exceptions.PageError:
        return f"'{query}'에 대한 위키백과 페이지를 찾을 수 없습니다."
    except Exception as e:
        return f"위키백과 검색 중 오류 발생: {e}"


def get_current_weather(city: str) -> str:
    """특정 도시의 현재 날씨를 알려줍니다. (Mock)"""
    city_lower = city.lower()
    weather_data = {
        "도쿄": "맑음, 28°C", "tokyo": "맑음, 28°C",
        "서울": "흐림, 24°C", "seoul": "흐림, 24°C",
        "뉴욕": "비, 18°C",   "new york": "비, 18°C",
    }
    for key, val in weather_data.items():
        if key in city_lower:
            return val
    return f"'{city}'의 날씨 정보를 찾을 수 없습니다."


# ── 함수 매핑 ────────────────────────────────────────────────
available_functions = {
    "get_exchange_rate": get_exchange_rate,
    "search_wikipedia":  search_wikipedia,
    "get_current_weather": get_current_weather
}

print("✅ Tool 함수 3개 정의 완료")

✅ Tool 함수 3개 정의 완료


In [4]:
# ── JSON Schema 정의 ─────────────────────────────────────────

exchange_schema = {
    "type": "function",
    "function": {
        "name": "get_exchange_rate",
        "description": "특정 통화와 대한민국 원(KRW) 사이의 실시간 환율을 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {"currency": {"type": "string", "description": "3자리 ISO 통화 코드 (예: USD, JPY, EUR)"}},
            "required": ["currency"]
        }
    }
}

wiki_schema = {
    "type": "function",
    "function": {
        "name": "search_wikipedia",
        "description": "위키백과에서 특정 주제에 대한 일반적인 사실 정보를 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "위키백과에서 검색할 키워드"}},
            "required": ["query"]
        }
    }
}

weather_schema = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "특정 도시의 현재 날씨를 알려줍니다.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"}},
            "required": ["city"]
        }
    }
}


all_schemas = [exchange_schema, wiki_schema, weather_schema]

print("✅ JSON Schema 3개 정의 완료")

✅ JSON Schema 3개 정의 완료


---
## ⚖️ 3. 비교: GPT 단독 실행 vs Agent

Agent 구조의 필요성을 체감하기 위해, 동일한 질문을 Tool 없이 일반 LLM에게만 물어봅니다.

In [5]:
question = "100달러는 지금 한국 원으로 얼마인가요?"

# ① GPT 단독 (Tool 없음)
gpt_only = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}]
).choices[0].message.content

# ② Agent (Tool 있음) – 3단계 수동 실행
messages = [{"role": "user", "content": question}]
resp = client.chat.completions.create(model=MODEL, messages=messages, tools=[exchange_schema], tool_choice="auto")
resp_msg = resp.choices[0].message
messages.append(resp_msg)

tc = resp_msg.tool_calls[0]
tool_result = get_exchange_rate(**json.loads(tc.function.arguments))
messages.append({"tool_call_id": tc.id, "role": "tool", "name": tc.function.name, "content": tool_result})

agent_answer = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content

print("===== ① GPT 단독 답변 (실시간 정보 없음) =====")
print(gpt_only)
print("\n===== ② Agent 답변 (실시간 환율 Tool 활용) =====")
print(agent_answer)

===== ① GPT 단독 답변 (실시간 정보 없음) =====
실시간 환율 정보를 제공할 수는 없지만, 일반적으로 환율은 변동성이 있기 때문에 정확한 금액은 환율 사이트나 은행에서 확인하는 것이 좋습니다. 예를 들어, 만약 1달러가 1,300원이라면 100달러는 130,000원이 됩니다. 현재 환율을 확인하시려면 인터넷에서 검색하거나 금융 기관에 문의하세요.

===== ② Agent 답변 (실시간 환율 Tool 활용) =====
현재 100달러는 약 151,665 한국 원입니다. (1달러 = 1,516.65원 기준)


> **결과 분석**: GPT 단독 실행은 실시간 정보에 접근할 수 없어 회피적인 답변을 합니다.  
> `get_exchange_rate` Tool을 사용한 Agent는 정확한 실시간 정보를 바탕으로 답변합니다.  
> 이것이 바로 외부 세계와 상호작용하는 **Agent가 필요한 이유**입니다.

---
## 🔁 4. Agent Full-Loop 수동 구현 ⭐

Agent가 실제로 동작하는 전 과정을 단계별로 나누어 직접 구현해봅니다.

```
1단계: LLM에 Tool 목록과 함께 질문 전달
     ↓
2단계: LLM이 Tool 호출 결정 → 실제 Tool 실행
     ↓
3단계: Tool 결과를 LLM에 전달 → 최종 답변 생성
```

In [6]:
# messages = [{"role": "user", "content": "현재 1유로(EUR)는 한국 원으로 얼마인가요?"}]
messages = [{"role": "user", "content": "서강대학교는 어디에 있나요?"}]
tools = [exchange_schema, wiki_schema]

print("💬 1단계: LLM에 Tool 호출 요청")
response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice="auto")
response_message = response.choices[0].message
messages.append(response_message)

tool_calls = response_message.tool_calls
if tool_calls:
    print(f"  → LLM이 Tool 사용을 결정했습니다.")
    print(f"  → 선택된 Tool: {tool_calls[0].function.name}")
    print(f"  → 인자: {tool_calls[0].function.arguments}")

    print("\n🔧 2단계: Tool 실행")
    for tc in tool_calls:
        fn_result = available_functions[tc.function.name](**json.loads(tc.function.arguments))
        print(f"  → 실행 결과: {fn_result}")
        messages.append({"tool_call_id": tc.id, "role": "tool", "name": tc.function.name, "content": fn_result})

    print("\n✍️ 3단계: 최종 답변 생성")
    final = client.chat.completions.create(model=MODEL, messages=messages)
    print("\n===== Agent의 최종 답변 =====")
    print(final.choices[0].message.content)
else:
    print("  → LLM이 Tool 사용을 결정하지 않았습니다.")
    print(response_message.content)

💬 1단계: LLM에 Tool 호출 요청
  → LLM이 Tool 사용을 결정했습니다.
  → 선택된 Tool: search_wikipedia
  → 인자: {"query":"서강대학교"}

🔧 2단계: Tool 실행
  → 실행 결과: 서강대학교(西江大學校, Sogang University, SU)는 대한민국의 로마 가톨릭 소속의 예수회가 설립한 사립 대학이다.
1960년 4월 18일에 예수회가 가톨릭 신앙을 바탕으로 설립하였다.

✍️ 3단계: 최종 답변 생성

===== Agent의 최종 답변 =====
서강대학교는 대한민국에 위치한 사립 대학으로, 로마 가톨릭 소속의 예수회가 설립하였습니다. 1960년 4월 18일에 설립되었습니다.


---
## 🤹 5. 심화: Multi-Tool 순차 실행

두 개의 Tool을 **순차적으로** 사용해야 하는 복잡한 문제를 수동으로 구현해봅니다.

> **도전 과제:** "일본의 수도는 어디인지 위키백과에서 찾고, 그 도시의 현재 날씨를 알려줘."

이 문제는 **정적인 지식(수도)** 과 **실시간 정보(날씨)** 를 결합해야 합니다.

In [7]:
print("--- 수동 Multi-Step Agent 실행 시작 ---")
tools = [wiki_schema, weather_schema]

# 1단계: 위키백과에서 일본의 수도 검색
print("\n--- 1단계: 수도 검색 ---")
messages = [{"role": "user", "content": "일본의 수도는 어디야?"}]
resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice="auto")
tc = resp.choices[0].message.tool_calls[0]
capital_info = available_functions[tc.function.name](**json.loads(tc.function.arguments))
print(f"결과: {capital_info}")

# 2단계: 결과에서 도시 이름만 추출 (LLM을 파서로 활용)
print("\n--- 2단계: 도시 이름 추출 ---")
messages = [
    {"role": "system", "content": "주어진 텍스트에서 도시 이름만 정확히 추출해서 알려줘. 다른 설명은 붙이지 마."},
    {"role": "user",   "content": capital_info}
]
capital_city = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content
print(f"추출된 도시: {capital_city}")

# 3단계: 추출된 도시로 날씨 검색
print("\n--- 3단계: 날씨 검색 ---")
messages = [{"role": "user", "content": capital_city + "의 현재 날씨 알려줘."}]
resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice="auto")
tc = resp.choices[0].message.tool_calls[0]
weather_info = available_functions[tc.function.name](**json.loads(tc.function.arguments))
print(f"결과: {weather_info}")

# 4단계: 모든 정보를 종합하여 최종 답변 생성
print("\n--- 4단계: 최종 답변 생성 ---")
final_prompt = f"다음 정보를 종합해서 한 문장으로 답변해줘.\n수도: {capital_city}\n날씨: {weather_info}"
final = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": final_prompt}])
print("\n===== 최종 답변 =====")
print(final.choices[0].message.content)

--- 수동 Multi-Step Agent 실행 시작 ---

--- 1단계: 수도 검색 ---
결과: 현재의 일본의 수도는 도쿄이다. 이 문서는 역사속 일본의 수도(日本の首都)에 관해 기술한 것이다.

--- 2단계: 도시 이름 추출 ---
추출된 도시: 도쿄

--- 3단계: 날씨 검색 ---
결과: 맑음, 28°C

--- 4단계: 최종 답변 생성 ---

===== 최종 답변 =====
현재 도쿄는 맑은 날씨에 28°C입니다.


> **고찰**: 성공했지만 LLM 호출을 **4번** 해야 했고, 중간 결과를 변수에 일일이 저장해야 했습니다.  
> 단계가 늘어날수록 코드는 더 복잡해집니다. 그리고 이 코드는 항상 성공하지 않습니다.



---
## 🤔 6. 수동 구현의 한계와 프레임워크의 필요성

지금까지 Agent의 핵심 로직을 '수동'으로 한 단계씩 구현했습니다. 더 복잡한 문제는 어떨까요?

| 문제점 | 내용 |
|:--- |:--- |
| **상태 관리 취약** | 중간 결과를 변수에 직접 저장 → 에러 시 전체 중단 |
| **단계 고정** | Tool 사용 횟수를 코드에 미리 결정해야 함 |
| **루프 없음** | 정보가 충분할 때까지 자동 반복 실행 불가 |
| **추적 어려움** | 어느 단계에서 무엇이 잘못됐는지 파악 어려움 |

```
"미국 초대 대통령은 누구인지 찾고,
 그가 태어난 도시의 현재 날씨를 알려줘."
```

이처럼 여러 단계를 거치고, 중간 결과를 저장하며, 다음 행동을 계획하는  
**상태 관리(State Management)와 실행 루프(Execution Loop)** 가 필요합니다.

이것을 직접 구현하는 것은 매우 번거롭습니다.  
이러한 복잡성을 해결해주는 것이 바로 **LangGraph** 입니다.

다음 파일(`2_langgraph_basic.ipynb`)에서는 오늘 수동으로 구현한 과정을  
LangGraph가 어떻게 자동화하고 단순화하는지 배웁니다.